## Imports

In [7]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local").appName("BigData").config("spark.ui.showConsoleProgress", "false").getOrCreate()


## Customers

In [8]:
df_customers = spark.read.csv("../data/0_raw/olist_customers_dataset.csv", header=True, inferSchema=True)

print("DF CUSTOMERS")
print("\n\n--- Dataframe preview")
df_customers.show(5)
print("\n\n--- Dataframe schema")
df_customers.printSchema()
print("\n\n---Number of lines :", df_customers.count())

df_customers.write.parquet("../data/1_bronze/customers.parquet", mode="overwrite")
print("\n\n✔ Dataset customers successfully exported in parquet format")

DF CUSTOMERS


--- Dataframe preview
+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------------+--------------+
|06b8999e2fba1a1fb...|861eff4711a542e4b...|                   14409|              franca|            SP|
|18955e83d337fd6b2...|290c77bc529b7ac93...|                    9790|sao bernardo do c...|            SP|
|4e7b3e00288586ebd...|060e732b5b29e8181...|                    1151|           sao paulo|            SP|
|b2b6027bc5c5109e5...|259dac757896d24d7...|                    8775|     mogi das cruzes|            SP|
|4f2d8ab171c80ec83...|345ecd01c38d18a90...|                   13056|            campinas|            SP|
+--------------------+--------------------+------------------------+--------------------+--------------+
only showing top 5

## Sellers

In [9]:
df_sellers = spark.read.csv("../data/0_raw/olist_sellers_dataset.csv", header=True, inferSchema=True)

print("DF SELLERS")
print("\n\n--- Dataframe preview")
df_sellers.show(5)
print("\n\n--- Dataframe schema")
df_sellers.printSchema()
print("\n\n---Number of lines :", df_sellers.count())

df_sellers.write.parquet("../data/1_bronze/sellers.parquet", mode="overwrite")
print("\n\n✔ Dataset sellers successfully exported in parquet format")

DF SELLERS


--- Dataframe preview
+--------------------+----------------------+-----------------+------------+
|           seller_id|seller_zip_code_prefix|      seller_city|seller_state|
+--------------------+----------------------+-----------------+------------+
|3442f8959a84dea7e...|                 13023|         campinas|          SP|
|d1b65fc7debc3361e...|                 13844|       mogi guacu|          SP|
|ce3ad9de960102d06...|                 20031|   rio de janeiro|          RJ|
|c0f3eea2e14555b6f...|                  4195|        sao paulo|          SP|
|51a04a8a6bdcb23de...|                 12914|braganca paulista|          SP|
+--------------------+----------------------+-----------------+------------+
only showing top 5 rows


--- Dataframe schema
root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: integer (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)



---Number of lines : 3095


✔ Data

## Geolocation

In [10]:
df_geo = spark.read.csv("../data/0_raw/olist_geolocation_dataset.csv", header=True, inferSchema=True)

print("DF GEOLOCATION")
print("\n\n--- Dataframe preview")
df_geo.show(5)
print("\n\n--- Dataframe schema")
df_geo.printSchema()
print("\n\n---Number of lines :", df_geo.count())

df_geo.write.parquet("../data/1_bronze/geolocation.parquet", mode="overwrite")
print("\n\n✔ Dataset geolocation successfully exported in parquet format")

DF GEOLOCATION


--- Dataframe preview
+---------------------------+-------------------+------------------+----------------+-----------------+
|geolocation_zip_code_prefix|    geolocation_lat|   geolocation_lng|geolocation_city|geolocation_state|
+---------------------------+-------------------+------------------+----------------+-----------------+
|                       1037| -23.54562128115268|-46.63929204800168|       sao paulo|               SP|
|                       1046|-23.546081127035535|-46.64482029837157|       sao paulo|               SP|
|                       1046| -23.54612896641469|-46.64295148361138|       sao paulo|               SP|
|                       1041|  -23.5443921648681|-46.63949930627844|       sao paulo|               SP|
|                       1035|-23.541577961711493|-46.64160722329613|       sao paulo|               SP|
+---------------------------+-------------------+------------------+----------------+-----------------+
only showing top 5 rows



## Join keys

In [13]:
# Loading remaining dataframes
df_items = spark.read.csv("../data/0_raw/olist_order_items_dataset.csv", header=True, inferSchema=True)
df_payments = spark.read.csv("../data/0_raw/olist_order_payments_dataset.csv", header=True, inferSchema=True)
df_reviews = spark.read.csv("../data/0_raw/olist_order_reviews_dataset.csv", header=True, inferSchema=True)
df_orders = spark.read.csv("../data/0_raw/olist_orders_dataset.csv", header=True, inferSchema=True)
df_products = spark.read.csv("../data/0_raw/olist_products_dataset.csv", header=True, inferSchema=True)
df_categories = spark.read.csv("../data/0_raw/product_category_name_translation.csv", header=True, inferSchema=True)

# Script to identify common columns in all dataframes
dataframes = {
    "items": df_items,
    "payments": df_payments,
    "reviews": df_reviews,
    "orders": df_orders,
    "products": df_products,
    "categories": df_categories,
    "customers": df_customers,
    "sellers": df_sellers,
    "geolocation": df_geo
}

columns = {}
for name, df in dataframes.items():
    for column in df.columns:
        if column not in columns:
            columns[column] = []
        columns[column].append(name)

# Display join keys with associated dataframes
print("JOIN KEYS :")

for column, dfs in sorted(columns.items()):
    if len(dfs) > 1:
        print(f"- {column} : {" - ".join(dfs)}")


JOIN KEYS :
- customer_id : orders - customers
- order_id : items - payments - reviews - orders
- product_category_name : products - categories
- product_id : items - products
- seller_id : items - sellers
